# Customer Segmentation Dashboard
Queries MySQL using SQLAlchemy, builds interactive charts with Plotly, and exports a self-contained HTML file.

In [13]:
# ── Cell 1: Imports & Connection ─────────────────────────────────────────────
import pandas as pd
from sqlalchemy import create_engine

# Update with your credentials
engine = create_engine("mysql+pymysql://root@localhost/ecommercedatabase")

print(engine)

Engine(mysql+pymysql://root@localhost/ecommercedatabase)


In [14]:
# ── Cell 2: Queries → DataFrames ─────────────────────────────────────────────

# Customer count and revenue by segment
df_segments = pd.read_sql("""
    SELECT
        cs.segment_name,
        COUNT(cp.profile_id)               AS customer_count,
        ROUND(SUM(cp.lifetime_value), 2)   AS total_revenue,
        ROUND(AVG(cp.avg_order_value), 2)  AS avg_order_value
    FROM customer_segments cs
    LEFT JOIN customer_profiles cp ON cs.segment_id = cp.segment_id
    GROUP BY cs.segment_name
    ORDER BY total_revenue DESC
""", engine)

# Average LTV by age band
df_age = pd.read_sql("""
    SELECT
        CASE
            WHEN age < 25 THEN 'Under 25'
            WHEN age BETWEEN 25 AND 34 THEN '25-34'
            WHEN age BETWEEN 35 AND 44 THEN '35-44'
            ELSE '45+'
        END AS age_band,
        ROUND(AVG(lifetime_value), 2) AS avg_ltv,
        COUNT(*) AS customer_count
    FROM customer_profiles
    GROUP BY age_band
    ORDER BY avg_ltv DESC
""", engine)

# Segment migration events
df_migrations = pd.read_sql("""
    SELECT
        old_seg.segment_name AS from_segment,
        new_seg.segment_name AS to_segment,
        COUNT(*)             AS event_count
    FROM segment_events se
    JOIN customer_segments old_seg ON se.old_segment_id = old_seg.segment_id
    JOIN customer_segments new_seg ON se.new_segment_id = new_seg.segment_id
    GROUP BY from_segment, to_segment
""", engine)

print('Queries complete')
print(df_segments)

Queries complete
  segment_name  customer_count  total_revenue  avg_order_value
0          VIP               4        8855.50           141.80
1        Loyal               6        4760.00           105.81
2      At Risk               3        1570.00            92.00
3   Occasional               3         925.00            93.33
4      Churned               4         905.00            60.29
5          New               5         488.99            97.80


In [15]:
# ── Cell 3: Build Charts & Export HTML ───────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['#2563EB', '#16A34A', '#CA8A04', '#DC2626', '#9333EA', '#0891B2']

# Chart 1 — Revenue by segment (bar)
fig1 = px.bar(
    df_segments,
    x='segment_name', y='total_revenue',
    title='Total Revenue by Segment',
    labels={'segment_name': 'Segment', 'total_revenue': 'Revenue ($)'},
    color='segment_name', color_discrete_sequence=colors,
    text='total_revenue'
)
fig1.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig1.update_layout(showlegend=False)

# Chart 2 — Customer count by segment (pie)
df_pie = df_segments[df_segments['customer_count'] > 0]
fig2 = px.pie(
    df_pie,
    names='segment_name', values='customer_count',
    title='Customer Distribution by Segment',
    color_discrete_sequence=colors
)

# Chart 3 — Avg LTV by age band (bar)
fig3 = px.bar(
    df_age,
    x='age_band', y='avg_ltv',
    title='Average Lifetime Value by Age Band',
    labels={'age_band': 'Age Band', 'avg_ltv': 'Avg LTV ($)'},
    color='age_band', color_discrete_sequence=colors,
    text='avg_ltv'
)
fig3.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig3.update_layout(showlegend=False)

# Chart 4 — Avg order value by segment (horizontal bar)
fig4 = px.bar(
    df_segments,
    x='avg_order_value', y='segment_name',
    orientation='h',
    title='Average Order Value by Segment',
    labels={'segment_name': 'Segment', 'avg_order_value': 'Avg Order Value ($)'},
    color='segment_name', color_discrete_sequence=colors
)
fig4.update_layout(showlegend=False)

# ── Combine into one HTML file ───────────────────────────────────────────────
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>Customer Segmentation Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; background: #f8fafc; margin: 0; padding: 20px; }}
        h1 {{ color: #1e293b; border-bottom: 3px solid #2563EB; padding-bottom: 10px; }}
        p {{ color: #475569; font-size: 15px; }}
        .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 20px; }}
        .card {{ background: white; border-radius: 8px; padding: 10px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
    </style>
</head>
<body>
    <h1>Customer Segmentation Dashboard</h1>
    <p>Analyzes customer distribution, lifetime value, and segment migration using MySQL + Python.</p>
    <div class='grid'>
        <div class='card'>{fig1.to_html(full_html=False, include_plotlyjs='cdn')}</div>
        <div class='card'>{fig2.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card'>{fig3.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card'>{fig4.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
</body>
</html>
"""

import os

output_path = os.path.expanduser('~/Downloads/segmentation_dashboard.html')

with open(output_path, 'w') as f:
    f.write(html_content)

print(f'Dashboard saved → {output_path}')

Dashboard saved → /Users/chenyuzhai/Downloads/segmentation_dashboard.html
